In [26]:
from pymongo import MongoClient
import os
import javalang
from typing import List

In [27]:
def connect_mongodb(uri, db_name, collection_name):
    # Connect to MongoDB 
    client = MongoClient(uri)
    
    # Select the database and collection
    db = client[db_name]
    collection = db[collection_name]
    
    return collection
def get_third_parties_with_stop_share(collection):
    # Query to find all documents where "stop_share" is "yes"
    results = collection.find({"stop_share": "yes"}, {"third_party": 1, "_id": 0})
    third_parties = [item["third_party"] for item in results]
    return third_parties
def get_files_with_extension(directory, extension, prefix=""):
    """
    Get all files in the given directory that start with a specific prefix
    and have a specific file extension.

    Args:
        directory (str): Path to the directory.
        extension (str): File extension (e.g., '.java').
        prefix (str): Optional prefix the filename must start with (e.g., 'wrap-').

    Returns:
        list: List of full file paths matching the extension and prefix.
    """
    return [
        os.path.join(directory, f)
        for f in os.listdir(directory)
        if os.path.isfile(os.path.join(directory, f))
        and f.lower().endswith(extension.lower())
        and f.startswith(prefix)
    ]
def extract_ast_paths(code: str) -> List[str]:
    try:
        tree = javalang.parse.parse(code)
    except javalang.parser.JavaSyntaxError as e:
        print(f"Error parsing Java code: {e}")
        return []
    
    paths = []
    
    def traverse(node, path):
        if isinstance(node, javalang.ast.Node):
            node_name = type(node).__name__
            path.append(node_name)
            
            for child in node.children:
                if isinstance(child, list):
                    for item in child:
                        traverse(item, path[:])
                else:
                    traverse(child, path[:])
                    
        elif isinstance(node, list):
            for item in node:
                traverse(item, path[:])
        else:
            if node is not None:
                paths.append(".".join(path) + " -> " + str(node))
    
    traverse(tree, [])
    return paths


# Read Java from file
def read_java_code_from_file(file_path: str) -> str:
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            code = file.read()
        return code
    except FileNotFoundError:
        print(f"File not found: {file_path}")
        return ""
    except Exception as e:
        print(f"Error reading file: {e}")
        return ""
def sanitize_code(code: str) -> str:
    return (
        code
        .replace("“", "\"")
        .replace("”", "\"")
        .replace("‘", "'")
        .replace("’", "'")
    )

In [28]:
# server
mongoDB_uri = 'mongodb://localhost:27017'
mongoDB_database = 'wearable-project-2' 
mongoDB_collection = 'third_party_services'
# direction
root_directory = r"C:\Users\ASUS\anaconda3-project-code\wearable-2-rag\rag-external-data"

In [29]:
# Connect to the MongoDB collection
collection = connect_mongodb(mongoDB_uri,mongoDB_database,mongoDB_collection)
third_parties_with_stop_share = get_third_parties_with_stop_share(collection)
print(third_parties_with_stop_share)
# print(len(third_parties_with_stop_share))

['google firebase', 'facebook', 'cleverTap', 'appLovin', 'kochava', 'braze', 'sentry', 'appsflyer', 'vungle', 'baidu', 'urban airship', 'branch', 'ironsource', 'ogury', 'moengage', 'Instabug', 'pushwoosh', 'bugsnag', 'onesignal', 'smaato', 'repro', 'batch', 'Taboola', 'Smart AdServer', 'Adjust', 'Samsung Health', 'Kakao', 'Heap Analytics', 'Digital Turbine', 'PubNative', 'Unity Ads', 'Braintree', 'Iterable', 'New Relic', 'Mixpanel', 'MobileFuse', 'chartbeat', 'swrve', 'Adobe', 'Naver', 'Adyen', 'HyprMX (Amazon)', 'Blueshift', 'optimizely', 'Chartboost', 'MParticle', 'UXCam', 'LogRocket', 'ACRA', 'Smartlook', 'Singular', 'Contentsquare', 'Localytics', 'CustomerIO', 'Sailthru']


In [30]:
for i in range(15, len(third_parties_with_stop_share)):
    print("---------------- Loop-"+str(i)+" ----------------")
    third_party_name = third_parties_with_stop_share[i]
    print("Third party name: ",third_party_name)
    directory_path = os.path.join(root_directory, third_party_name)
    print("Directory path: ",directory_path)
    file_arr = get_files_with_extension(directory_path, ".java","wrap-")
    print("file_arr: ",file_arr)
    for j in range(len(file_arr)):
        code_file_path = file_arr[j]
        print("Code file path: ",code_file_path)
        # Read JAVA from file
        java_code = read_java_code_from_file(code_file_path)
        sanitize_java_code = sanitize_code(java_code)
        # Create AST
        if java_code:
            ast_paths = extract_ast_paths(sanitize_java_code)
            save_ast_path = directory_path+"//"+"ast-"+str(j)+".txt"
            # Write to file
            with open(save_ast_path, "w", encoding='utf-8') as f:
                for path in ast_paths:
                    f.write(path + "\n")
                    
            print("AST paths have been saved")
    # break

---------------- Loop-15 ----------------
Third party name:  Instabug
Directory path:  C:\Users\ASUS\anaconda3-project-code\wearable-2-rag\rag-external-data\Instabug
file_arr:  ['C:\\Users\\ASUS\\anaconda3-project-code\\wearable-2-rag\\rag-external-data\\Instabug\\wrap-java-0.java']
Code file path:  C:\Users\ASUS\anaconda3-project-code\wearable-2-rag\rag-external-data\Instabug\wrap-java-0.java
Error parsing Java code: 
AST paths have been saved
---------------- Loop-16 ----------------
Third party name:  pushwoosh
Directory path:  C:\Users\ASUS\anaconda3-project-code\wearable-2-rag\rag-external-data\pushwoosh
file_arr:  ['C:\\Users\\ASUS\\anaconda3-project-code\\wearable-2-rag\\rag-external-data\\pushwoosh\\wrap-java-0.java']
Code file path:  C:\Users\ASUS\anaconda3-project-code\wearable-2-rag\rag-external-data\pushwoosh\wrap-java-0.java
Error parsing Java code: 
AST paths have been saved
---------------- Loop-17 ----------------
Third party name:  bugsnag
Directory path:  C:\Users\ASU